<a href="https://colab.research.google.com/github/Satyam-Mittal2527/PyTorch/blob/main/pytorch_training_pipeline_improced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [5]:
df.shape

(569, 33)

In [6]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [7]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [8]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

## Numpy arrays to PyTorch Tensors

In [11]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

## Defining the model

In [12]:
class MySimpleNN():

  def __init__(self, X):
    self.weights = torch.rand(X.shape[1], 1, dtype = torch.float64, requires_grad = True)
    self.bias = torch.zeros(1, dtype = torch.float64, requires_grad = True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


## important Parameters

In [13]:
learning_rate = 0.1
epochs = 25

## Training Pipeline

In [14]:
model = MySimpleNN(X_train_tensor)
#define loop
for epoch in range(epochs):

  #Forward Pass
  y_pred = model.forward(X_train_tensor)

  #loss calculation
  loss = model.loss_function(y_pred, y_train_tensor)

  #backward pass
  loss.backward()

  #Update parameters
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # Zero gradient
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  #print loss in each epoch
  print(f"Epoch: {epoch +1}, loss : {loss.item()}")

Epoch: 1, loss : 3.5355066442539753
Epoch: 2, loss : 3.412580412115022
Epoch: 3, loss : 3.2849796665591096
Epoch: 4, loss : 3.151453968326769
Epoch: 5, loss : 3.0094239500865423
Epoch: 6, loss : 2.860185918915845
Epoch: 7, loss : 2.7006901868529085
Epoch: 8, loss : 2.538495863639449
Epoch: 9, loss : 2.367577095525813
Epoch: 10, loss : 2.194928323528224
Epoch: 11, loss : 2.0217582496424984
Epoch: 12, loss : 1.8508495259178719
Epoch: 13, loss : 1.6853899901919884
Epoch: 14, loss : 1.528596654147007
Epoch: 15, loss : 1.3822040561383777
Epoch: 16, loss : 1.2513073722701025
Epoch: 17, loss : 1.1378687391350264
Epoch: 18, loss : 1.0432082821821143
Epoch: 19, loss : 0.9675265057081907
Epoch: 20, loss : 0.9095461486670992
Epoch: 21, loss : 0.8666453102224095
Epoch: 22, loss : 0.8355082096818206
Epoch: 23, loss : 0.8128827725575681
Epoch: 24, loss : 0.7960755977026988
Epoch: 25, loss : 0.7831106429631521


## Model Evaluation

In [15]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6015697121620178


## NN Module implementation

In [16]:
import torch.nn as nn
class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()

    self.network = nn.Sequential(
        nn.Linear(num_features,1),
        nn.Sigmoid()
    )
  def forward(self, X):
    y_pred = self.network(X)

    return y_pred


In [17]:
loss_function = nn.BCELoss()

In [18]:
model2 = MySimpleNN(X_train_tensor.shape[1])

#define optimizer
optimizer = torch.optim.SGD(model2.parameters(), lr = learning_rate)

X_train_tensor = X_train_tensor.float()
y_train_tensor = y_train_tensor.float()
X_test_tensor = X_test_tensor.float()
for epoch in range(epochs):

  y_pred = model2(X_train_tensor)

  #loss calculation
  loss = loss_function(y_pred, y_train_tensor.reshape(-1,1))

  #Clear gradients
  optimizer.zero_grad()

  #backward
  loss.backward()

  #Update parameters
  optimizer.step()


  #print loss in each epoch
  print(f"Epoch: {epoch +1}, loss : {loss.item()}")



Epoch: 1, loss : 0.9127132892608643
Epoch: 2, loss : 0.6383143067359924
Epoch: 3, loss : 0.5026131272315979
Epoch: 4, loss : 0.42825889587402344
Epoch: 5, loss : 0.3807986378669739
Epoch: 6, loss : 0.3472757637500763
Epoch: 7, loss : 0.3219841718673706
Epoch: 8, loss : 0.30201399326324463
Epoch: 9, loss : 0.28571414947509766
Epoch: 10, loss : 0.2720714211463928
Epoch: 11, loss : 0.2604256868362427
Epoch: 12, loss : 0.250326544046402
Epoch: 13, loss : 0.2414548695087433
Epoch: 14, loss : 0.23357747495174408
Epoch: 15, loss : 0.22651953995227814
Epoch: 16, loss : 0.22014707326889038
Epoch: 17, loss : 0.2143554389476776
Epoch: 18, loss : 0.20906150341033936
Epoch: 19, loss : 0.20419830083847046
Epoch: 20, loss : 0.1997111290693283
Epoch: 21, loss : 0.19555479288101196
Epoch: 22, loss : 0.1916915625333786
Epoch: 23, loss : 0.1880895495414734
Epoch: 24, loss : 0.1847216933965683
Epoch: 25, loss : 0.18156468868255615


In [19]:
with torch.no_grad():
  y_pred = model2.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5461680293083191


## Adding Dataset and DataLoader class

In [33]:
from torch.utils.data import Dataset, DataLoader

In [34]:
class CustomDataset(Dataset):

  def __init__(self, features, label):
    self.features = features
    self.label = label

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index], self.label[index]

In [35]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [36]:
train_dataset[10]

(tensor([ 2.2661,  0.4222,  2.3892,  2.5533, -0.1633,  1.7857,  1.6971,  1.5961,
         -0.3214, -0.1238,  3.1116,  0.2196,  3.4379,  3.2844, -0.4465,  1.3501,
          0.9773,  0.5976, -0.1685,  0.3123,  2.5370,  0.4365,  2.7315,  2.7410,
         -0.2111,  1.4800,  1.5504,  1.0961, -0.0953,  0.2926]),
 tensor(1.))

In [37]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [38]:
model3= MySimpleNN(X_train_tensor.shape[1])
for epoch in range(epochs):
  for batch_features, batch_label in train_loader:
    y_pred = model3(batch_features)

    loss = loss_function(y_pred, batch_label.view(-1,1))
    optimizer.zero_grad()

      #backward
    loss.backward()

      #Update parameters
    optimizer.step()


    #print loss in each epoch
  print(f"Epoch: {epoch +1}, loss : {loss.item()}")



Epoch: 1, loss : 0.7633780837059021
Epoch: 2, loss : 0.7208883166313171
Epoch: 3, loss : 0.7378453612327576
Epoch: 4, loss : 0.8349027633666992
Epoch: 5, loss : 0.7465768456459045
Epoch: 6, loss : 0.6389853358268738
Epoch: 7, loss : 0.7019752860069275
Epoch: 8, loss : 0.7294412851333618
Epoch: 9, loss : 0.6496686339378357
Epoch: 10, loss : 0.8505934476852417
Epoch: 11, loss : 0.7493513226509094
Epoch: 12, loss : 0.690001904964447
Epoch: 13, loss : 0.748374342918396
Epoch: 14, loss : 0.6238284111022949
Epoch: 15, loss : 0.6310938596725464
Epoch: 16, loss : 0.8686155676841736
Epoch: 17, loss : 0.7187595963478088
Epoch: 18, loss : 0.7755336761474609
Epoch: 19, loss : 0.5709618926048279
Epoch: 20, loss : 0.7517356276512146
Epoch: 21, loss : 0.7380765080451965
Epoch: 22, loss : 0.79070645570755
Epoch: 23, loss : 0.6075342893600464
Epoch: 24, loss : 0.7563653588294983
Epoch: 25, loss : 0.7247396111488342


In [39]:
model3.eval()
accuracy_list=[]
with torch.no_grad():
  for batch_features, batch_label in test_loader:
    y_pred = model3.forward(batch_features)
    y_pred = (y_pred > 0.9).float()
    batch_accuracy = (y_pred.view(-1) == batch_label).float().mean().item()
    print(f'Accuracy: {accuracy.item()}')
    accuracy_list.append(batch_accuracy)
overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print(f"Final Accuarcy:{overall_accuracy}")

Accuracy: 0.5461680293083191
Accuracy: 0.5461680293083191
Accuracy: 0.5461680293083191
Accuracy: 0.5461680293083191
Final Accuarcy:0.6180555522441864
